In [ ]:
# ============================================================
# PHASE 3 RETRAIN (FROM CACHED FEATURES)
# NeuroScreen - SSBD BiLSTM
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIG
# ============================================================

BASE = "/content/drive/MyDrive/NeuroScreen"

FEATURE_X = BASE + "/ssbd2/X_features.npy"
FEATURE_Y = BASE + "/ssbd2/y_labels.npy"

CHECKPOINT = BASE + "/checkpoints/behavior_checkpoint.pth"
BEST_MODEL = BASE + "/models/behavior_model_best.pth"

os.makedirs(BASE + "/checkpoints", exist_ok=True)
os.makedirs(BASE + "/models", exist_ok=True)
os.makedirs(BASE + "/results", exist_ok=True)

BATCH_SIZE = 16
LR = 1e-3
EPOCHS_THIS_SESSION = 20

SEED = 42

# ============================================================
# SEED
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("="*60)
print("DEVICE")
print("="*60)
print("Using:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# LOAD FEATURES
# ============================================================

print("\nLoading cached features...")

X = np.load(FEATURE_X)
y = np.load(FEATURE_Y)

print("X shape:", X.shape)
print("y shape:", y.shape)

NUM_CLASSES = len(np.unique(y))

# ============================================================
# STRATIFIED SPLIT
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print("\nSplit")
print("Train:", len(X_train))
print("Val  :", len(X_val))

# ============================================================
# DATASET
# ============================================================

class PoseDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = PoseDataset(X_train, y_train)
val_dataset = PoseDataset(X_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ============================================================
# MODEL
# ============================================================

INPUT_SIZE = X.shape[2]

class BehaviorLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=INPUT_SIZE,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(
            128 * 2,
            NUM_CLASSES
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.dropout(out)

        out = self.fc(out)

        return out

model = BehaviorLSTM().to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

# ============================================================
# RESUME
# ============================================================

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT):

    ckpt = torch.load(
        CHECKPOINT,
        map_location=DEVICE
    )

    model.load_state_dict(
        ckpt["model_state_dict"]
    )

    optimizer.load_state_dict(
        ckpt["optimizer_state_dict"]
    )

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    print("\nCheckpoint found.")
    print("Resuming from epoch:", start_epoch)

else:
    print("\nNo checkpoint found.")

# ============================================================
# EVALUATION
# ============================================================

def evaluate():

    model.eval()

    preds_all = []
    labels_all = []

    total_loss = 0

    with torch.no_grad():

        for x, yb in val_loader:

            x = x.to(DEVICE)
            yb = yb.to(DEVICE)

            outputs = model(x)

            loss = criterion(outputs, yb)

            total_loss += loss.item()

            preds = torch.argmax(
                outputs,
                dim=1
            )

            preds_all.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                yb.cpu().numpy()
            )

    acc = accuracy_score(
        labels_all,
        preds_all
    )

    macro_f1 = f1_score(
        labels_all,
        preds_all,
        average='macro'
    )

    return (
        total_loss / len(val_loader),
        acc,
        macro_f1
    )

# ============================================================
# TRAIN
# ============================================================

history = []

print("\n" + "="*60)
print("TRAINING")
print("="*60)

for epoch in range(
    start_epoch,
    start_epoch + EPOCHS_THIS_SESSION
):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    t0 = time.time()

    for x, yb in train_loader:

        x = x.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(
            outputs,
            yb
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(
            outputs,
            dim=1
        )

        total += yb.size(0)

        correct += (
            preds == yb
        ).sum().item()

    train_acc = correct / total

    val_loss, val_acc, val_f1 = evaluate()

    scheduler.step(val_acc)

    elapsed = (
        time.time() - t0
    ) / 60

    print(f"\nEpoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f}")
    print(f"Val Acc  : {val_acc:.4f}")
    print(f"Val F1   : {val_f1:.4f}")
    print(f"Time     : {elapsed:.2f} min")

    history.append({
        "epoch": epoch+1,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "best_acc":
                best_acc
        },
        CHECKPOINT
    )

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            BEST_MODEL
        )

        print(">>> BEST MODEL SAVED")

# ============================================================
# SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(history)

history_path = (
    BASE +
    "/results/behavior_training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

print("\n" + "="*60)
print("DONE")
print("="*60)

print("Best Val Acc:", best_acc)

print("\nCheckpoint:")
print(CHECKPOINT)

print("\nBest model:")
print(BEST_MODEL)

print("\nHistory:")
print(history_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEVICE
Using: cuda
GPU: Tesla T4

Loading cached features...
X shape: (183, 30, 132)
y shape: (183,)

Split
Train: 146
Val  : 37

No checkpoint found.

TRAINING

Epoch 1
Train Acc: 0.4178
Val Acc  : 0.6216
Val F1   : 0.6239
Time     : 0.02 min
>>> BEST MODEL SAVED

Epoch 2
Train Acc: 0.7466
Val Acc  : 0.6486
Val F1   : 0.6457
Time     : 0.00 min
>>> BEST MODEL SAVED

Epoch 3
Train Acc: 0.7808
Val Acc  : 0.6216
Val F1   : 0.6053
Time     : 0.00 min

Epoch 4
Train Acc: 0.8219
Val Acc  : 0.6216
Val F1   : 0.6255
Time     : 0.00 min

Epoch 5
Train Acc: 0.6644
Val Acc  : 0.7027
Val F1   : 0.6836
Time     : 0.00 min
>>> BEST MODEL SAVED

Epoch 6
Train Acc: 0.7534
Val Acc  : 0.5676
Val F1   : 0.5529
Time     : 0.00 min

Epoch 7
Train Acc: 0.7945
Val Acc  : 0.7568
Val F1   : 0.7442
Time     : 0.00 min
>>> BEST MODEL SAVED

Epoch 8
Train Acc: 0.8630
Val Acc  : 0.7568


In [ ]:
import os
import re
import json
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score

from google.colab import drive

# ============================================================
# DRIVE
# ============================================================

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/NeuroScreen"

FEATURES_X = BASE + "/ssbd2/X_features.npy"
FEATURES_Y = BASE + "/ssbd2/y_labels.npy"

VIDEO_ROOT = BASE + "/ssbd2"

CHECKPOINT = BASE + "/checkpoints/behavior_group_checkpoint.pth"
BEST_MODEL = BASE + "/models/behavior_group_best.pth"

os.makedirs(BASE + "/checkpoints", exist_ok=True)
os.makedirs(BASE + "/models", exist_ok=True)

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# ============================================================
# LOAD FEATURES
# ============================================================

X = np.load(FEATURES_X)
y = np.load(FEATURES_Y)

print("X:", X.shape)
print("y:", y.shape)

# ============================================================
# REBUILD GROUP IDS FROM VIDEOS
# ============================================================

video_paths = []

for cls in ["armFlapping", "headBanging", "spinning"]:

    cls_root = os.path.join(VIDEO_ROOT, cls)

    for root, _, files in os.walk(cls_root):

        for f in sorted(files):

            if f.endswith(".mp4"):
                video_paths.append(os.path.join(root, f))

print("Videos discovered:", len(video_paths))

assert len(video_paths) == len(X)

groups = []

for path in video_paths:

    fname = os.path.basename(path)

    m = re.match(r"(\d+)", fname)

    if m:
        gid = m.group(1)
    else:
        gid = fname

    groups.append(gid)

groups = np.array(groups)

print("Unique groups:", len(np.unique(groups)))

# ============================================================
# GROUP SPLIT
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(X, y, groups)
)

X_train = X[train_idx]
y_train = y[train_idx]

X_val = X[val_idx]
y_val = y[val_idx]

print()
print("Train:", len(X_train))
print("Val  :", len(X_val))

# ============================================================
# DATASET
# ============================================================

class PoseDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(
    PoseDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    PoseDataset(X_val, y_val),
    batch_size=32,
    shuffle=False
)

# ============================================================
# MODEL
# ============================================================

class PoseLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=132,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Sequential(
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,3)
        )

    def forward(self,x):

        out,_ = self.lstm(x)

        out = out[:,-1,:]

        return self.fc(out)

model = PoseLSTM().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# ============================================================
# LOAD CHECKPOINT
# ============================================================

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT):

    ckpt = torch.load(CHECKPOINT)

    model.load_state_dict(
        ckpt["model"]
    )

    optimizer.load_state_dict(
        ckpt["optimizer"]
    )

    start_epoch = ckpt["epoch"]

    best_acc = ckpt["best_acc"]

    print(
        f"Resuming from epoch {start_epoch}"
    )

# ============================================================
# TRAIN
# ============================================================

history = []

for epoch in range(start_epoch, start_epoch + 20):

    t0 = time.time()

    model.train()

    train_preds = []
    train_true = []

    for Xb,yb in train_loader:

        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(Xb)

        loss = criterion(logits,yb)

        loss.backward()

        optimizer.step()

        pred = logits.argmax(1)

        train_preds.extend(
            pred.cpu().numpy()
        )

        train_true.extend(
            yb.cpu().numpy()
        )

    train_acc = accuracy_score(
        train_true,
        train_preds
    )

    # VALIDATION

    model.eval()

    val_preds = []
    val_true = []

    with torch.no_grad():

        for Xb,yb in val_loader:

            Xb = Xb.to(device)
            yb = yb.to(device)

            logits = model(Xb)

            pred = logits.argmax(1)

            val_preds.extend(
                pred.cpu().numpy()
            )

            val_true.extend(
                yb.cpu().numpy()
            )

    val_acc = accuracy_score(
        val_true,
        val_preds
    )

    val_f1 = f1_score(
        val_true,
        val_preds,
        average="macro"
    )

    print(
        f"Epoch {epoch+1} | "
        f"Train={train_acc:.4f} | "
        f"Val={val_acc:.4f} | "
        f"F1={val_f1:.4f}"
    )

    torch.save(
        {
            "epoch":epoch+1,
            "model":model.state_dict(),
            "optimizer":optimizer.state_dict(),
            "best_acc":best_acc
        },
        CHECKPOINT
    )

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            BEST_MODEL
        )

        print(">>> BEST MODEL SAVED")

    history.append([
        epoch+1,
        train_acc,
        val_acc,
        val_f1
    ])

pd.DataFrame(
    history,
    columns=[
        "epoch",
        "train_acc",
        "val_acc",
        "val_f1"
    ]
).to_csv(
    BASE + "/results/behavior_group_history.csv",
    index=False
)

print()
print("Finished")
print("Best Val Acc:", best_acc)
print("Model:", BEST_MODEL)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
X: (183, 30, 132)
y: (183,)
Videos discovered: 183
Unique groups: 75

Train: 146
Val  : 37
Epoch 1 | Train=0.3356 | Val=0.4324 | F1=0.3531
>>> BEST MODEL SAVED
Epoch 2 | Train=0.4041 | Val=0.6757 | F1=0.6140
>>> BEST MODEL SAVED
Epoch 3 | Train=0.5342 | Val=0.8108 | F1=0.8060
>>> BEST MODEL SAVED
Epoch 4 | Train=0.7260 | Val=0.8919 | F1=0.8775
>>> BEST MODEL SAVED
Epoch 5 | Train=0.7192 | Val=0.7568 | F1=0.7500
Epoch 6 | Train=0.6781 | Val=0.9189 | F1=0.9109
>>> BEST MODEL SAVED
Epoch 7 | Train=0.7055 | Val=0.8919 | F1=0.8775
Epoch 8 | Train=0.7945 | Val=0.7297 | F1=0.6854
Epoch 9 | Train=0.7466 | Val=0.6486 | F1=0.6103
Epoch 10 | Train=0.8288 | Val=0.8919 | F1=0.8918
Epoch 11 | Train=0.8562 | Val=0.7297 | F1=0.6896
Epoch 12 | Train=0.8562 | Val=0.8378 | F1=0.8016
Epoch 13 | Train=0.8699 | Val=0.9189 | F1=0.9109
Epoch 14 | Train=0.8904 | Val=0.83